# Learn GMM from demonstrations

Config → load CSV → featurize → fit → export JSON for C++ GMR.

Demos live in `assets/demonstrations/demo_*.csv` (`demo_id,t,<outputs...>`).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Allow imports when the notebook cwd is python/
sys.path.insert(0, str(Path.cwd()))

from util import MODEL_DIR, load_demo_csv, output_dim
from gmm_learning import build_feature_matrix, export_model, fit_gmm

# --- config ---
DEMO = "synthetic"          # bottle_neck | synthetic | time_traj
N_STATES = 5
MODE = "pos"                # pos | pos_vel | pos_vel_acc
OUT = MODEL_DIR / "model_synthetic.json"
RANDOM_STATE = 0

In [ ]:
demos = load_demo_csv(DEMO)
X, meta = build_feature_matrix(demos, mode=MODE)
gmm = fit_gmm(X, N_STATES, random_state=RANDOM_STATE)
out_path = export_model(gmm, OUT, meta)

print(f"demos={meta['n_demos']}  D={meta['output_dim']}  feature_dim={meta['dim']}  mode={MODE}")
print(f"states={N_STATES}  wrote {out_path}")
print(f"Mu shape={gmm.means_.shape}  Sigma shape={gmm.covariances_.shape}")

In [ ]:
D = output_dim(demos)
fig, axes = plt.subplots(1, 3 if D >= 2 else 2, figsize=(15, 4.5))
axes = np.atleast_1d(axes)

if D >= 2:
    ax_xy, ax_tx, ax_ty = axes[:3]
    for i, d in enumerate(demos):
        ax_xy.plot(d["y"][:, 0], d["y"][:, 1], lw=1.5, label=f"demo {i}")
        ax_tx.plot(d["t"], d["y"][:, 0], lw=1.5)
        ax_ty.plot(d["t"], d["y"][:, 1], lw=1.5)
    # GMM means in (t, y) feature space for MODE=pos: cols [t, y0, y1, ...]
    if MODE == "pos" and gmm.means_.shape[1] >= 3:
        ax_xy.scatter(gmm.means_[:, 1], gmm.means_[:, 2], c="k", s=40, zorder=5, label="GMM μ")
    ax_xy.set_title(f"{DEMO}: x–y")
    ax_xy.set_aspect("equal", adjustable="datalim")
    ax_xy.legend(fontsize=8)
    ax_xy.grid(True, alpha=0.3)
    ax_tx.set_title("t–x"); ax_tx.grid(True, alpha=0.3)
    ax_ty.set_title("t–y"); ax_ty.grid(True, alpha=0.3)
else:
    ax = axes[0]
    for i, d in enumerate(demos):
        ax.plot(d["t"], d["y"][:, 0], lw=1.5, label=f"demo {i}")
    ax.set_title(f"{DEMO}: t–y0")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"GMM learning: {DEMO} ({MODE})")
fig.tight_layout()
plt.show()